# Short-Horizon Return Prediction - Clean Pipeline
Complete Kaggle submission notebook (memory-optimized)

## Cell 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import catboost as cb
import gc
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("✓ All libraries imported successfully!")

## Cell 2: Load and Optimize Data

In [ ]:
def optimize_dtypes(df):
    """Compress data types to save memory"""
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != 'object':
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

print("Loading training data...")
train_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet')
train_df = optimize_dtypes(train_df)

print("Loading test data...")
test_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet')
test_df = optimize_dtypes(test_df)

print(f"✓ Train shape: {train_df.shape}")
print(f"✓ Test shape: {test_df.shape}")
print(f"✓ Train memory: {train_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"✓ Test memory: {test_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## Cell 3: Data Cleaning and Feature Engineering

In [ ]:
# Check what we have
print(f"Train columns: {train_df.shape[1]}, Test columns: {test_df.shape[1]}")
print(f"Train missing: {train_df.isnull().sum().sum()}, Test missing: {test_df.isnull().sum().sum()}")

# Fill NaNs with median
train_df = train_df.fillna(train_df.median())
test_df = test_df.fillna(test_df.median())
print(f"✓ Missing values filled")

# Extract target and IDs BEFORE feature cleaning
y_train = train_df['TARGET'].values.copy()
train_ids = train_df['ID'].values.copy()
test_ids = test_df['ID'].values.copy()

# Remove ID and TARGET from features
X_train = train_df.drop(['ID', 'TARGET'], axis=1).copy()
X_test = test_df.drop(['ID'], axis=1).copy()

print(f"✓ Extracted target: {y_train.shape}")
print(f"✓ Initial features - Train: {X_train.shape}, Test: {X_test.shape}")

# Remove constant columns from BOTH datasets
const_cols_train = list(X_train.columns[X_train.nunique() <= 1])
const_cols_test = list(X_test.columns[X_test.nunique() <= 1])
const_cols_all = list(set(const_cols_train + const_cols_test))

if const_cols_all:
    X_train = X_train.drop(const_cols_all, axis=1, errors='ignore')
    X_test = X_test.drop(const_cols_all, axis=1, errors='ignore')
    print(f"✓ Removed {len(const_cols_all)} constant columns")

# Ensure same features in train and test
common_cols = sorted(list(set(X_train.columns) & set(X_test.columns)))
X_train = X_train[common_cols].copy()
X_test = X_test[common_cols].copy()

print(f"✓ Final features - Train: {X_train.shape}, Test: {X_test.shape}")
print(f"✓ Features match: {X_train.shape[1] == X_test.shape[1]}")

# Clean up memory
del train_df, test_df
gc.collect()

print(f"\nTarget Statistics:")
print(f"  Mean: {y_train.mean():.4f}")
print(f"  Std: {y_train.std():.4f}")
print(f"  Min: {y_train.min():.4f}")
print(f"  Max: {y_train.max():.4f}")

## Cell 4: Validation Setup

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# 3-fold CV for memory efficiency
kf = KFold(n_splits=3, shuffle=True, random_state=42)
fold_indices = list(kf.split(X_train))

print(f"✓ Using 3-Fold Cross-Validation")
print(f"✓ Fold sizes: ~{len(X_train)//3:,} per fold")

# Initialize storage for OOF predictions
oof_lgb = np.zeros(len(X_train))
oof_cb = np.zeros(len(X_train))
oof_ridge = np.zeros(len(X_train))

test_pred_lgb = np.zeros(len(X_test))
test_pred_cb = np.zeros(len(X_test))
test_pred_ridge = np.zeros(len(X_test))

lgb_scores = []
cb_scores = []
ridge_scores = []

print(f"✓ Storage initialized for {len(X_train)} train + {len(X_test)} test")

## Cell 5: LightGBM Training

In [ ]:
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'num_leaves': 20,
    'learning_rate': 0.05,
    'max_depth': 5,
    'min_data_in_leaf': 30,
    'lambda_l1': 1.0,
    'lambda_l2': 1.0,
    'feature_fraction': 0.6,
    'bagging_fraction': 0.6,
    'random_state': 42,
    'verbose': -1,
    'num_threads': 4
}

for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
    print(f"\nLightGBM Fold {fold_num}/3...")
    
    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]
    y_tr = y_train[train_idx]
    y_val = y_train[val_idx]
    
    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val)
    
    model = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=500,
        valid_sets=[val_data],
        callbacks=[
            lgb.early_stopping(30),
            lgb.log_evaluation(period=0)
        ]
    )
    
    oof_lgb[val_idx] = model.predict(X_val)
    test_pred_lgb += model.predict(X_test) / 3
    
    r2 = r2_score(y_val, oof_lgb[val_idx])
    lgb_scores.append(r2)
    print(f"  R² = {r2:.6f}")
    
    del model, X_tr, X_val, y_tr, y_val, train_data, val_data
    gc.collect()

print(f"\n✓ LightGBM Mean R²: {np.mean(lgb_scores):.6f} (±{np.std(lgb_scores):.6f})")

## Cell 6: CatBoost Training

In [ ]:
cb_params = {
    'iterations': 300,
    'depth': 4,
    'learning_rate': 0.05,
    'l2_leaf_reg': 5.0,
    'random_state': 42,
    'verbose': False,
    'task_type': 'CPU',
    'thread_count': 4
}

for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
    print(f"\nCatBoost Fold {fold_num}/3...")
    
    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]
    y_tr = y_train[train_idx]
    y_val = y_train[val_idx]
    
    model = cb.CatBoostRegressor(**cb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=20,
        verbose=False
    )
    
    oof_cb[val_idx] = model.predict(X_val)
    test_pred_cb += model.predict(X_test) / 3
    
    r2 = r2_score(y_val, oof_cb[val_idx])
    cb_scores.append(r2)
    print(f"  R² = {r2:.6f}")
    
    del model, X_tr, X_val, y_tr, y_val
    gc.collect()

print(f"\n✓ CatBoost Mean R²: {np.mean(cb_scores):.6f} (±{np.std(cb_scores):.6f})")

## Cell 7: Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
    print(f"Ridge Fold {fold_num}/3...")
    
    X_tr = X_train_scaled[train_idx]
    X_val = X_train_scaled[val_idx]
    y_tr = y_train[train_idx]
    y_val = y_train[val_idx]
    
    model = Ridge(alpha=1.0, random_state=42)
    model.fit(X_tr, y_tr)
    
    oof_ridge[val_idx] = model.predict(X_val)
    test_pred_ridge += model.predict(X_test_scaled) / 3
    
    r2 = r2_score(y_val, oof_ridge[val_idx])
    ridge_scores.append(r2)
    print(f"  R² = {r2:.6f}")

print(f"\n✓ Ridge Mean R²: {np.mean(ridge_scores):.6f} (±{np.std(ridge_scores):.6f})")

## Cell 8: Ensembling

In [ ]:
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)

lgb_mean = np.mean(lgb_scores)
cb_mean = np.mean(cb_scores)
ridge_mean = np.mean(ridge_scores)

print(f"LightGBM:  R² = {lgb_mean:.6f} (±{np.std(lgb_scores):.6f})")
print(f"CatBoost:  R² = {cb_mean:.6f} (±{np.std(cb_scores):.6f})")
print(f"Ridge:     R² = {ridge_mean:.6f} (±{np.std(ridge_scores):.6f})")

# Compute ensemble weights
total = lgb_mean + cb_mean + ridge_mean
w_lgb = lgb_mean / total
w_cb = cb_mean / total
w_ridge = ridge_mean / total

print(f"\nEnsemble Weights:")
print(f"  LightGBM: {w_lgb*100:.1f}%")
print(f"  CatBoost: {w_cb*100:.1f}%")
print(f"  Ridge:    {w_ridge*100:.1f}%")

# Create ensemble
oof_ensemble = w_lgb * oof_lgb + w_cb * oof_cb + w_ridge * oof_ridge
ensemble_r2 = r2_score(y_train, oof_ensemble)

print(f"\nEnsemble OOF R²: {ensemble_r2:.6f}")
print("="*60)

## Cell 9: Test Predictions

In [ ]:
# Ensemble test predictions
test_pred_final = w_lgb * test_pred_lgb + w_cb * test_pred_cb + w_ridge * test_pred_ridge

print(f"Test Predictions Stats:")
print(f"  Mean: {test_pred_final.mean():.6f}")
print(f"  Std:  {test_pred_final.std():.6f}")
print(f"  Min:  {test_pred_final.min():.6f}")
print(f"  Max:  {test_pred_final.max():.6f}")

print(f"\nTraining Target Stats (for comparison):")
print(f"  Mean: {y_train.mean():.6f}")
print(f"  Std:  {y_train.std():.6f}")
print(f"  Min:  {y_train.min():.6f}")
print(f"  Max:  {y_train.max():.6f}")

## Cell 10: Create Submission

In [ ]:
# Create submission CSV
submission = pd.DataFrame({
    'ID': test_ids,
    'TARGET': test_pred_final
})

print(f"Submission shape: {submission.shape}")
print(f"\nFirst 5 rows:")
print(submission.head())

# Check for any NaN in submission
nan_count = submission.isnull().sum().sum()
print(f"\nNaN values in submission: {nan_count}")

# Save submission
submission.to_csv('submission.csv', index=False)
print(f"\n✓✓✓ Submission saved as 'submission.csv' ✓✓✓")

## Cell 11: Final Summary

In [ ]:
print("\n" + "="*70)
print("COMPETITION SUBMISSION COMPLETE")
print("="*70)
print(f"\nDataset:")
print(f"  Training samples: {len(X_train):,}")
print(f"  Test samples: {len(X_test):,}")
print(f"  Features used: {X_train.shape[1]}")
print(f"\nModel Performance (3-Fold CV):")
print(f"  LightGBM:  {lgb_mean:.6f}")
print(f"  CatBoost:  {cb_mean:.6f}")
print(f"  Ridge:     {ridge_mean:.6f}")
print(f"  Ensemble:  {ensemble_r2:.6f}")
print(f"\nSubmission:")
print(f"  File: submission.csv")
print(f"  Rows: {len(submission)}")
print(f"  Columns: {submission.columns.tolist()}")
print(f"\nNext Steps:")
print(f"  1. Download submission.csv")
print(f"  2. Submit to Kaggle")
print(f"  3. Monitor leaderboard")
print("\n" + "="*70)